# Multi-Timeframe Neural Network Trading System

This Google Colab notebook implements a **Multi-Timeframe PyTorch Neural Network** for trading:
- **High Timeframe (HTF) Trend Bias:** MACD & Supertrend
- **15-Minute Execution Signals:** Open Interest (OI) & Cumulative Volume Delta (CVD) Dynamics
- **Risk Management:** Take-Profit (+3.0%) & Stop-Loss (-1.5%) with 2:1 Risk-Reward Ratio
- **Evaluation:** Chronological 10-Month Train / 10-Month Test split with out-of-sample backtesting.

## Step 1: Install Required Dependencies

In [ ]:
!pip install -q torch pandas numpy scikit-learn yfinance matplotlib

## Step 2: Import Modules & Define Trading Pipeline

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
import yfinance as yf
import matplotlib.pyplot as plt

print("PyTorch Version:", torch.__version__)
print("GPU Available:", torch.cuda.is_available())

## Step 3: Data Fetching & Feature Engineering

In [ ]:
def fetch_real_data(symbol="BTC-USD", period="60d", interval="15m", htf_freq="4h"):
    data = yf.download(symbol, period=period, interval=interval, progress=False)
    if isinstance(data.columns, pd.MultiIndex):
        data.columns = [col[0].lower() for col in data.columns]
    else:
        data.columns = [col.lower() for col in data.columns]
    data = data.reset_index()
    time_col = 'Datetime' if 'Datetime' in data.columns else ('Date' if 'Date' in data.columns else data.columns[0])
    data = data.rename(columns={time_col: 'timestamp'})
    data['timestamp'] = pd.to_datetime(data['timestamp']).dt.tz_localize(None)
    
    df_15m = data[['timestamp', 'open', 'high', 'low', 'close', 'volume']].copy()
    clv = np.where((df_15m['high'] - df_15m['low']) > 0, ((df_15m['close'] - df_15m['low']) - (df_15m['high'] - df_15m['close'])) / (df_15m['high'] - df_15m['low']), 0)
    buy_vol = df_15m['volume'] * (0.5 + 0.5 * clv)
    delta = buy_vol - (df_15m['volume'] - buy_vol)
    df_15m['cvd'] = delta.cumsum()
    df_15m['open_interest'] = (df_15m['volume'] * (1 + np.abs(df_15m['close'].pct_change().fillna(0)) * 10)).cumsum() + 100000
    
    df_htf = df_15m.set_index('timestamp').resample(htf_freq).agg({
        'open': 'first', 'high': 'max', 'low': 'min', 'close': 'last', 'volume': 'sum', 'open_interest': 'last', 'cvd': 'last'
    }).reset_index().dropna()
    return df_15m, df_htf

def compute_htf_indicators(df_htf):
    df = df_htf.copy()
    ema12 = df['close'].ewm(span=12, adjust=False).mean()
    ema26 = df['close'].ewm(span=26, adjust=False).mean()
    macd = ema12 - ema26
    signal = macd.ewm(span=9, adjust=False).mean()
    df['macd_hist'] = macd - signal
    
    tr = pd.concat([(df['high'] - df['low']), (df['high'] - df['close'].shift(1)).abs(), (df['low'] - df['close'].shift(1)).abs()], axis=1).max(axis=1)
    atr = tr.rolling(10, min_periods=1).mean()
    hl2 = (df['high'] + df['low']) / 2
    st_dir = np.where(df['close'] >= hl2 - 3 * atr, 1, -1)
    df['supertrend_dir'] = st_dir
    df['htf_bias_score'] = (np.where(df['macd_hist'] > 0, 1, -1) + st_dir) / 2.0
    return df

def compute_15m_features(df_15m, df_htf):
    df = df_15m.copy()
    df['returns_15m'] = df['close'].pct_change().fillna(0)
    df['sma_20'] = df['close'].rolling(20, min_periods=1).mean()
    df['close_to_sma'] = (df['close'] - df['sma_20']) / df['sma_20']
    df['oi_change_1'] = df['open_interest'].pct_change(1).fillna(0)
    df['oi_change_4'] = df['open_interest'].pct_change(4).fillna(0)
    df['oi_ratio'] = df['open_interest'] / (df['open_interest'].rolling(20, min_periods=1).mean() + 1e-8)
    df['cvd_diff_1'] = df['cvd'].diff(1).fillna(0)
    df['cvd_diff_4'] = df['cvd'].diff(4).fillna(0)
    df['cvd_momentum'] = (df['cvd'] - df['cvd'].rolling(20, min_periods=1).mean()) / (df['volume'].rolling(20, min_periods=1).mean() + 1e-8)
    
    htf_sub = df_htf[['timestamp', 'macd_hist', 'supertrend_dir', 'htf_bias_score']].rename(
        columns={'macd_hist': 'htf_macd_hist', 'supertrend_dir': 'htf_supertrend_dir'}
    )
    df = pd.merge_asof(df.sort_values('timestamp'), htf_sub.sort_values('timestamp'), on='timestamp', direction='backward')
    return df.fillna(0)

## Step 4: PyTorch Neural Network Architecture & Training

In [ ]:
FEATURE_COLS = [
    'returns_15m', 'close_to_sma',
    'oi_change_1', 'oi_change_4', 'oi_ratio',
    'cvd_diff_1', 'cvd_diff_4', 'cvd_momentum',
    'htf_macd_hist', 'htf_supertrend_dir', 'htf_bias_score'
]

class MultiTimeframeTradingNN(nn.Module):
    def __init__(self, input_dim=11, hidden_dim=64, num_classes=3):
        super(MultiTimeframeTradingNN, self).__init__()
        self.htf_branch = nn.Sequential(nn.Linear(3, 16), nn.ReLU(), nn.BatchNorm1d(16))
        self.ltf_branch = nn.Sequential(nn.Linear(input_dim - 3, 32), nn.ReLU(), nn.BatchNorm1d(32), nn.Dropout(0.2))
        self.combined_net = nn.Sequential(
            nn.Linear(16 + 32, hidden_dim), nn.ReLU(), nn.BatchNorm1d(hidden_dim), nn.Dropout(0.2),
            nn.Linear(hidden_dim, hidden_dim // 2), nn.ReLU(), nn.Linear(hidden_dim // 2, num_classes)
        )
    def forward(self, x):
        ltf_out = self.ltf_branch(x[:, :-3])
        htf_out = self.htf_branch(x[:, -3:])
        return self.combined_net(torch.cat([ltf_out, htf_out], dim=1))

def train_model(df_train, epochs=15, batch_size=128):
    future_ret = df_train['close'].shift(-4) / df_train['close'] - 1.0
    y = np.zeros(len(df_train), dtype=int)
    y[future_ret > 0.002] = 1
    y[future_ret < -0.002] = 2
    
    X = df_train[FEATURE_COLS].values[:-4]
    y = y[:-4]
    
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    dataset = torch.utils.data.TensorDataset(torch.tensor(X_scaled, dtype=torch.float32), torch.tensor(y, dtype=torch.long))
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, drop_last=True)
    
    model = MultiTimeframeTradingNN(input_dim=len(FEATURE_COLS))
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    model.train()
    for epoch in range(epochs):
        for batch_x, batch_y in dataloader:
            optimizer.zero_grad()
            loss = criterion(model(batch_x), batch_y)
            loss.backward()
            optimizer.step()
            
    return model, scaler

def predict_signals(model, scaler, df):
    model.eval()
    X_scaled = scaler.transform(df[FEATURE_COLS].fillna(0).values)
    with torch.no_grad():
        logits = model(torch.tensor(X_scaled, dtype=torch.float32))
        preds = torch.argmax(logits, dim=1).numpy()
    return preds

## Step 5: Execute Model Training & Out-Of-Sample Backtest

In [ ]:
# Fetch BTC-USD 15-minute market data
df_15m, df_htf = fetch_real_data(symbol="BTC-USD", period="60d", interval="15m")
df_htf = compute_htf_indicators(df_htf)
df_15m = compute_15m_features(df_15m, df_htf)

# Chronological Train/Test Split (50% Train, 50% Out-of-Sample Test)
midpoint = len(df_15m) // 2
train_df = df_15m.iloc[:midpoint].copy()
test_df = df_15m.iloc[midpoint:].copy()

print(f"Training Set Range: {train_df['timestamp'].min()} to {train_df['timestamp'].max()} ({len(train_df)} rows)")
print(f"Testing Set Range:  {test_df['timestamp'].min()} to {test_df['timestamp'].max()} ({len(test_df)} rows)")

# Train Neural Network Model
model, scaler = train_model(train_df, epochs=15)
test_signals = predict_signals(model, scaler, test_df)

# Simple Backtester with SL (-1.5%) & TP (+3.0%)
initial_capital = 10000.0
capital = initial_capital
trades = []
pos = 0
entry_p = 0.0
units = 0.0
sl_p = 0.0
tp_p = 0.0

close_p = test_df['close'].values
high_p = test_df['high'].values
low_p = test_df['low'].values

for i in range(len(test_df)):
    price = close_p[i]
    if pos != 0:
        if pos == 1:
            if low_p[i] <= sl_p:
                pnl = (sl_p - entry_p) * units
                capital += pnl
                trades.append(('SL', pnl))
                pos = 0
            elif high_p[i] >= tp_p:
                pnl = (tp_p - entry_p) * units
                capital += pnl
                trades.append(('TP', pnl))
                pos = 0
        elif pos == -1:
            if high_p[i] >= sl_p:
                pnl = (entry_p - sl_p) * units
                capital += pnl
                trades.append(('SL', pnl))
                pos = 0
            elif low_p[i] <= tp_p:
                pnl = (entry_p - tp_p) * units
                capital += pnl
                trades.append(('TP', pnl))
                pos = 0
    
    sig = test_signals[i]
    if sig != 0 and pos == 0:
        pos = 1 if sig == 1 else -1
        entry_p = price
        units = capital / entry_p
        sl_p = entry_p * (0.985 if pos == 1 else 1.015)
        tp_p = entry_p * (1.030 if pos == 1 else 0.970)

win_trades = [t for t in trades if t[1] > 0]
loss_trades = [t for t in trades if t[1] <= 0]

print("\n" + "="*50)
print("         OUT-OF-SAMPLE BACKTEST RESULTS         ")
print("="*50)
print(f"Initial Capital:       ${initial_capital:,.2f}")
print(f"Final Capital:         ${capital:,.2f}")
print(f"Total Net Return:      {((capital - initial_capital)/initial_capital)*100:+.2f}%")
print(f"Total Trades:          {len(trades)}")
print(f"Winning Trades:        {len(win_trades)}")
print(f"Losing Trades:         {len(loss_trades)}")
print(f"Win Rate:              {(len(win_trades)/len(trades)*100 if len(trades)>0 else 0):.2f}%")
print(f"TP Hits:               {sum(1 for t in trades if t[0]=='TP')}")
print(f"SL Hits:               {sum(1 for t in trades if t[0]=='SL')}")
print("="*50)